# Lab 18: Principal Component Analysis

## Finding the coordinate system hidden inside data

This lab is a full computational companion to Chapter 18.  PCA will appear here as geometry, statistics, projection, compression, denoising, and visualization.

You will work through:

1. a 2D data cloud with a visible principal direction;
2. centering and covariance;
3. variance along directions;
4. eigenvector PCA;
5. SVD PCA;
6. explained variance;
7. scaling effects;
8. PCA for 3D-to-2D visualization;
9. PCA for denoising;
10. PCA in high dimensions.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

rng = np.random.default_rng(18)

## 1. A data cloud with a hidden direction

We start with a two-dimensional cloud. It looks like a stretched ellipse. PCA should discover the direction of that stretch.

In [ ]:
n = 350
t = rng.normal(0, 2.5, n)
noise = rng.normal(0, 0.55, n)
X = np.column_stack([t, 0.7*t + noise]) + np.array([3.0, -1.0])

plt.figure(figsize=(6,6))
plt.scatter(X[:,0], X[:,1], alpha=0.45)
plt.scatter([X[:,0].mean()], [X[:,1].mean()], marker='x', s=100)
plt.title('Original data cloud')
plt.xlabel('feature 1')
plt.ylabel('feature 2')
plt.grid(True)
plt.axis('equal')
plt.show()

## 2. Center the cloud

PCA studies variation around the center.  Subtracting the mean moves the cloud so that its center is the origin.

In [ ]:
mean = X.mean(axis=0)
Xc = X - mean

print('Mean vector:', mean)
print('Mean after centering:', Xc.mean(axis=0))

plt.figure(figsize=(6,6))
plt.scatter(Xc[:,0], Xc[:,1], alpha=0.45)
plt.axhline(0)
plt.axvline(0)
plt.title('Centered data cloud')
plt.xlabel('centered feature 1')
plt.ylabel('centered feature 2')
plt.grid(True)
plt.axis('equal')
plt.show()

## 3. Variance along a direction

For a unit vector $u$, the projection coordinates are $X_cu$.  The variance in direction $u$ is the average squared coordinate.

In [ ]:
def unit(theta):
    return np.array([np.cos(theta), np.sin(theta)])

def variance_along(Xc, u):
    return np.mean((Xc @ u)**2)

angles = np.linspace(0, np.pi, 361)
vars_along = np.array([variance_along(Xc, unit(a)) for a in angles])
best = angles[np.argmax(vars_along)]

plt.figure(figsize=(7,4))
plt.plot(angles, vars_along)
plt.axvline(best, linestyle='--')
plt.xlabel('direction angle in radians')
plt.ylabel('variance along direction')
plt.title('PCA searches for the direction with largest projected variance')
plt.grid(True)
plt.show()

print('Best angle:', best)
print('Max variance:', vars_along.max())

## 4. Covariance matrix and eigenvectors

The covariance matrix stores how features vary together. PCA directions are eigenvectors of this matrix.

In [ ]:
C = Xc.T @ Xc / len(Xc)
print('Covariance matrix:
', C)

evals, evecs = np.linalg.eigh(C)
idx = np.argsort(evals)[::-1]
evals = evals[idx]
evecs = evecs[:, idx]
print('Eigenvalues:', evals)
print('Eigenvectors as columns:
', evecs)

plt.figure(figsize=(6,6))
plt.scatter(Xc[:,0], Xc[:,1], alpha=0.35)
for j in range(2):
    v = evecs[:, j]
    length = 2*np.sqrt(evals[j])
    plt.arrow(0, 0, length*v[0], length*v[1], head_width=0.15, length_includes_head=True)
    plt.arrow(0, 0, -length*v[0], -length*v[1], head_width=0.15, length_includes_head=True)
plt.axhline(0)
plt.axvline(0)
plt.grid(True)
plt.axis('equal')
plt.title('Principal component directions')
plt.show()

## 5. Scores: coordinates in the PCA language

The loading vectors are directions. The scores are the coordinates of each point in those directions.

In [ ]:
Y = Xc @ evecs
print(Y[:5])

plt.figure(figsize=(6,6))
plt.scatter(Y[:,0], Y[:,1], alpha=0.45)
plt.axhline(0)
plt.axvline(0)
plt.xlabel('PC1 score')
plt.ylabel('PC2 score')
plt.title('Same data in PCA coordinates')
plt.grid(True)
plt.axis('equal')
plt.show()

print('Covariance in PCA coordinates:
', Y.T @ Y / len(Y))

## 6. PCA reconstruction from one component

Keeping one component projects the data cloud onto its best one-dimensional shadow.

In [ ]:
V1 = evecs[:, :1]
Y1 = Xc @ V1
Xc_hat_1 = Y1 @ V1.T
X_hat_1 = Xc_hat_1 + mean

plt.figure(figsize=(6,6))
plt.scatter(X[:,0], X[:,1], alpha=0.25, label='original')
plt.scatter(X_hat_1[:,0], X_hat_1[:,1], alpha=0.65, label='rank-1 PCA reconstruction')
for i in range(0, n, 25):
    plt.plot([X[i,0], X_hat_1[i,0]], [X[i,1], X_hat_1[i,1]], linewidth=0.8)
plt.legend()
plt.grid(True)
plt.axis('equal')
plt.title('Projection onto the first principal component')
plt.show()

mse1 = np.mean((X - X_hat_1)**2)
print('Reconstruction MSE using 1 component:', mse1)

## 7. PCA via SVD

PCA is SVD applied to centered data.  The right singular vectors are the principal directions.

In [ ]:
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
print('Singular values:', S)
print('Variances from SVD:', S**2 / len(Xc))
print('Right singular vectors V:
', Vt.T)
print('Compare absolute directions with covariance eigenvectors:
', np.abs(Vt.T), '
', np.abs(evecs))

## 8. Explained variance

The explained variance ratio says how much of the total variation each component captures.

In [ ]:
explained = evals / evals.sum()
print('Explained variance ratio:', explained)
print('Cumulative:', np.cumsum(explained))

plt.figure(figsize=(6,4))
plt.bar([1,2], explained)
plt.plot([1,2], np.cumsum(explained), marker='o')
plt.xticks([1,2])
plt.xlabel('component')
plt.ylabel('fraction of variance')
plt.title('Explained variance')
plt.ylim(0,1.05)
plt.grid(True)
plt.show()

## 9. Scaling can change PCA

Here one feature is artificially multiplied by a large number. PCA may then chase scale rather than structure.

In [ ]:
X_scaled_bad = X.copy()
X_scaled_bad[:,1] *= 20
Xb = X_scaled_bad - X_scaled_bad.mean(axis=0)
Cb = Xb.T @ Xb / len(Xb)
eb, vb = np.linalg.eigh(Cb)
idx = np.argsort(eb)[::-1]
eb, vb = eb[idx], vb[:,idx]

X_standard = (X - X.mean(axis=0)) / X.std(axis=0)
Cs = X_standard.T @ X_standard / len(X_standard)
es, vs = np.linalg.eigh(Cs)
idx = np.argsort(es)[::-1]
es, vs = es[idx], vs[:,idx]

print('PCA direction after multiplying feature 2 by 20:', vb[:,0])
print('PCA direction after standardization:', vs[:,0])

## 10. A 3D cloud projected to 2D

PCA is often used for visualization.  Here we make a 3D cloud that mostly lives near a 2D plane, then project it to two principal components.

In [ ]:
m = 700
a = rng.normal(0, 2.0, m)
b = rng.normal(0, 0.8, m)
c = 0.2*rng.normal(size=m)
X3 = np.column_stack([a + 0.2*b, 0.5*a + b, -0.3*a + 0.8*b + c])
X3c = X3 - X3.mean(axis=0)
U3, S3, Vt3 = np.linalg.svd(X3c, full_matrices=False)
Y3 = X3c @ Vt3.T
expl3 = (S3**2) / np.sum(S3**2)
print('Explained variance ratios:', expl3)

fig = plt.figure(figsize=(7,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X3[:,0], X3[:,1], X3[:,2], alpha=0.3, s=10)
ax.set_title('Original 3D cloud')
plt.show()

plt.figure(figsize=(6,5))
plt.scatter(Y3[:,0], Y3[:,1], alpha=0.35, s=12)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('2D PCA projection')
plt.grid(True)
plt.axis('equal')
plt.show()

## 11. PCA for denoising

We create data from a low-dimensional signal plus noise. PCA can recover the dominant structure by discarding low-variance directions.

In [ ]:
n, d, r = 300, 40, 3
latent = rng.normal(size=(n, r))
basis = rng.normal(size=(r, d))
X_clean = latent @ basis
X_noisy = X_clean + 1.0*rng.normal(size=(n, d))
Xn = X_noisy - X_noisy.mean(axis=0)
U, S, Vt = np.linalg.svd(Xn, full_matrices=False)

errors = []
ks = range(1, 21)
for k in ks:
    Xk = (U[:,:k] * S[:k]) @ Vt[:k,:] + X_noisy.mean(axis=0)
    errors.append(np.mean((X_clean - Xk)**2))

plt.figure(figsize=(7,4))
plt.plot(list(ks), errors, marker='o')
plt.xlabel('number of PCA components kept')
plt.ylabel('MSE compared with clean signal')
plt.title('PCA denoising: keeping too few or too many components can hurt')
plt.grid(True)
plt.show()

plt.figure(figsize=(7,4))
plt.plot(S**2 / np.sum(S**2), marker='o')
plt.xlabel('component index')
plt.ylabel('explained variance ratio')
plt.title('Scree plot for noisy low-rank data')
plt.grid(True)
plt.show()

## 12. High-dimensional PCA experiment

The final experiment shows PCA discovering a hidden low-dimensional subspace inside high-dimensional data.

In [ ]:
n, d, r = 500, 120, 5
Z = rng.normal(size=(n, r))
W = rng.normal(size=(r, d))
X_hd = Z @ W + 0.35*rng.normal(size=(n, d))
X_hd -= X_hd.mean(axis=0)
U, S, Vt = np.linalg.svd(X_hd, full_matrices=False)
expl = S**2 / np.sum(S**2)

plt.figure(figsize=(7,4))
plt.plot(np.cumsum(expl[:30]), marker='o')
plt.xlabel('number of components')
plt.ylabel('cumulative explained variance')
plt.title('High-dimensional data with hidden low-dimensional structure')
plt.ylim(0,1.02)
plt.grid(True)
plt.show()

print('Cumulative explained variance first 5 components:', np.cumsum(expl)[4])

## Student reflection

Write short answers to these questions:

1. What did centering do geometrically?
2. What is the difference between a loading and a score?
3. How does PCA relate to projection?
4. How does PCA relate to SVD?
5. Give one example where PCA is useful and one example where PCA may be misleading.

## Optional extension

Apply PCA to a real dataset from your own research, teaching, or interests. Create a two-page summary with one scree plot, one PCA visualization, and a careful interpretation of the first two loading vectors.